# 第 11 章 · 推理工程(精简版)

> 本文是 [ch11.ipynb](./ch11.ipynb) 的浓缩版。只保留核心 schema 和关键流程,方便快速复习。

## OpenAI API 请求/响应 Schema

**请求**:
```json
{
  "model": "minimind",
  "messages": [{"role": "user", "content": "你好"}],
  "stream": true,
  "tools": [...],
  "open_thinking": false
}
```

**非流式响应**:
```json
{
  "choices": [{
    "message": {
      "role": "assistant",
      "content": "你好!",
      "reasoning_content": "<think> 内容",
      "tool_calls": [...]
    },
    "finish_reason": "stop" | "tool_calls"
  }]
}
```

## SSE 流式 Chunk 格式

```
data: {"choices":[{"delta":{"content":"你"}}]}
data: {"choices":[{"delta":{"content":"好"}}]}
data: {"choices":[{"delta":{"reasoning_content":"思考中..."}}]}
data: {"choices":[{"delta":{"tool_calls":[...]}}]}
data: {"choices":[{"delta":{},"finish_reason":"stop"}]}
```

三层管道:`model.generate → CustomStreamer → Queue → StreamingResponse`

## `<think>` / `<tool_call>` 解析规则

| 模型原始输出 | 解析后 |
|---|---|
| `<think>推理内容</think>` | `message.reasoning_content` |
| `<tool_call>{"name":"f","arguments":{}}</tool_call>` | `message.tool_calls` |
| 剩余正文 | `message.content` |

解析函数 `parse_response`(serve_openai_api.py:83-102)用正则提取,失败则跳过。

## 模型格式转换

```
训练侧 (.pth)                     部署侧
MiniMindForCausalLM    convert     Qwen3ForCausalLM
  state_dict        ─────────→     HF transformers
                                    │
                          ┌─────────┼─────────┐
                         vllm     ollama    llama.cpp
```

**关键映射**:`MiniMindConfig` → `Qwen3Config`(字段一一对应)

**MoE expert stacking**:8 个独立 expert → 1 个 stacked tensor `(num_experts, ...)`

**LoRA merge**:$W' = W + BA$,合并后丢弃 LoRA,部署用合并版

## 下一步

推理工程 → 第 12 章:DPO 对齐